# Fourth PyCUDA example: vector addition with ElementwiseKernel

This fourth notebook computes the **same vector sum**, with the same input values, $N=100003$ and `float32` precision as the preceding examples.

`ElementwiseKernel` lets us specify the argument types and the operation for one array element, while PyCUDA generates the surrounding CUDA kernel and manages its launch. We keep the input and output arrays on the GPU and reuse the output allocation.

The final optional extension retains the original notebook's linear combination $c_i=2a_i+3b_i$. It is presented separately so the main example remains a direct continuation of the vector-addition sequence.

Run the cells in order in a GPU runtime. Results and timings appear inline; no result files are written.


## 1. Prepare the GPU runtime

In Google Colab, select **Runtime → Change runtime type → GPU** and connect. The following commands inspect the assigned GPU and CUDA compiler on the remote machine. `SourceModule` needs `nvcc`; a GPU runtime alone does not establish that the compiler is available.


In [1]:
!nvidia-smi
!nvcc --version


Wed Oct 14 17:45:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Install PyCUDA if necessary. The installation may build the package and take several minutes. Continue only after installation succeeds.


In [2]:
!pip install pycuda


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 21.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 11.7 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp313-cp313-linux_x86_64.whl size=5281389 sha256=3e273c08d46b64249bb07986f719f33a14f9358704933be0a0b3123371377771
  Stored in directory: /root/.cache/pip/wheels/ce/26/46/c519675fcb0e5e17bab8e85b6676528c40d12d794182340e85
Successfully built pycuda


## 2. Initialize PyCUDA

Use the same context initialization and environment printout as before. Import `ElementwiseKernel` from `pycuda.elementwise`, in addition to the `gpuarray` module. PyCUDA still needs a working CUDA compiler to build the generated kernel.


In [3]:
import sys
import numpy as np
import pycuda
import pycuda.driver as cuda
import pycuda.autoinit
import pycuda.gpuarray as gpuarray
from pycuda.elementwise import ElementwiseKernel

device = cuda.Context.get_device()
print('GPU:', device.name())
print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('PyCUDA:', pycuda.VERSION_TEXT)


GPU: Tesla T4
Python: 3.13.15
NumPy: 2.1.3
PyCUDA: 2026.1


## 3. Choose the vector length

Use $N=100003$ as before. The generated kernel covers the vector without a user-defined block size or grid. The arrays in this example all have the same length.


In [4]:
N = 100003
assert 1 <= N <= np.iinfo(np.int32).max
print(f'Elements: {N}')
print('Launch configuration: managed by ElementwiseKernel.')


Elements: 100003
Launch configuration: managed by ElementwiseKernel.


## 4. Prepare the host arrays

Use the same deterministic inputs as before: $a_i=0.25(i\bmod101)$ and $b_i=0.50(i\bmod37)$. Both arrays have type `np.float32`. We allocate `h_c` on the host for the result to be retrieved later.


In [5]:
indices = np.arange(N, dtype=np.int32)
h_a = np.float32(0.25) * (indices % 101).astype(np.float32)
h_b = np.float32(0.50) * (indices % 37).astype(np.float32)
h_c = np.empty_like(h_a)

assert h_a.dtype == h_b.dtype == h_c.dtype == np.float32
assert all(x.flags.c_contiguous for x in (h_a, h_b, h_c))
print(f'Bytes per vector: {h_a.nbytes}')


Bytes per vector: 400012


## 5. Transfer the inputs and allocate the output

`gpuarray.to_gpu` creates the input arrays on the GPU and copies the host data. Allocate `d_c` once with `gpuarray.empty_like`: the kernel will write into this existing output buffer at every call.

Unlike `d_c = d_a + d_b` in the third example, calling our `ElementwiseKernel` does not create a new output array. The output is one of its arguments. It need not be initialized because the operation writes every element.


In [6]:
d_a = gpuarray.to_gpu(h_a)
d_b = gpuarray.to_gpu(h_b)
d_c = gpuarray.empty_like(d_a)

assert d_a.shape == d_b.shape == d_c.shape == (N,)
assert d_a.dtype == d_b.dtype == d_c.dtype == np.float32
print('Device array class:', type(d_a).__name__)
print('Shape:', d_a.shape)
print('Dtype:', d_a.dtype)
print('Bytes per device array:', d_a.nbytes)


Device array class: GPUArray
Shape: (100003,)
Dtype: float32
Bytes per device array: 400012


## 6. Define the operation for one element

The first string declares the arguments using C syntax. The second string gives the CUDA C operation for index `i`. This string is compiled for the GPU; it is not a Python loop executed on the CPU.

PyCUDA supplies the index `i`, the iteration over valid elements, and the launch configuration. We therefore do not write `__global__`, `threadIdx`, a bounds check, or a complete `SourceModule` kernel. `const` marks the two inputs as read-only within this operation.

The generated kernel computes the same sum as the Chapter 2 kernel, but its generated implementation need not be identical to that source.


In [7]:
elementwise_add = ElementwiseKernel(
    "const float *a, const float *b, float *c",
    "c[i] = a[i] + b[i]",
    "vector_add_elementwise",
)


## 7. Apply the operation to the device arrays

Pass the arrays in the declared order: two inputs followed by the output. No length or block/grid arguments are supplied: PyCUDA obtains the iteration size from the arrays. We have explicitly checked their matching shapes and types above.

The call submits GPU work. The blocking retrieval in the next cell ensures that the result is ready for use on the CPU.


In [8]:
elementwise_add(d_a, d_b, d_c)


## 8. Retrieve and validate every output

Copy the result into `h_c` using `get`. Check every element against the same NumPy reference used in the first two examples, rejecting nonfinite values and using the same tolerances.


In [9]:
d_c.get(ary=h_c)
h_reference = h_a + h_b

if not np.all(np.isfinite(h_c)):
    raise AssertionError('The GPU result contains nonfinite values.')
np.testing.assert_allclose(h_c, h_reference, rtol=1e-6, atol=1e-6)
max_abs_error = float(np.max(np.abs(h_c - h_reference)))
print(f'Validation passed for all {N} elements.')
print(f'Maximum absolute error: {max_abs_error:.3e}')


Validation passed for all 100003 elements.
Maximum absolute error: 0.000e+00


## 9. Display the computed sum

Display the first twelve and the last three entries using the same columns as before. The GPU column contains values actually retrieved from `d_c`.


In [10]:
shown_indices = sorted(set(range(min(12, N))) | set(range(max(0, N - 3), N)))
print(f"{'i':>8} {'a[i]':>12} {'b[i]':>12} {'c[i] (GPU)':>15} {'NumPy':>12}")
for i in shown_indices:
    print(f'{i:8d} {h_a[i]:12.6f} {h_b[i]:12.6f} '
          f'{h_c[i]:15.6f} {h_reference[i]:12.6f}')


       i         a[i]         b[i]      c[i] (GPU)        NumPy
       0     0.000000     0.000000        0.000000     0.000000
       1     0.250000     0.500000        0.750000     0.750000
       2     0.500000     1.000000        1.500000     1.500000
       3     0.750000     1.500000        2.250000     2.250000
       4     1.000000     2.000000        3.000000     3.000000
       5     1.250000     2.500000        3.750000     3.750000
       6     1.500000     3.000000        4.500000     4.500000
       7     1.750000     3.500000        5.250000     5.250000
       8     2.000000     4.000000        6.000000     6.000000
       9     2.250000     4.500000        6.750000     6.750000
      10     2.500000     5.000000        7.500000     7.500000
      11     2.750000     5.500000        8.250000     8.250000
  100000     2.500000    13.000000       15.500000    15.500000
  100001     2.750000    13.500000       16.250000    16.250000
  100002     3.000000    14.000000      

## 10. Optional: measure the device interval

The arrays are already allocated and the inputs remain on the GPU. After three warm-up calls and synchronization, CUDA events bracket 100 calls to the generated kernel. Every call writes into the same `d_c` allocation.

As in the first notebook, report the mean device interval per repeated launch. Allocation, compilation, host-device transfers, validation and printing are excluded. For a short kernel, the interval may include device idle gaps between Python submissions; it is not guaranteed to equal pure instruction-execution time.

This timing scope follows the first notebook. It differs from the second notebook's complete calls including transfers and from the third notebook's host-timed expressions including output allocation. No speedup between those different scopes is claimed.


In [11]:
REPEATS = 100
assert REPEATS > 0

for _ in range(3):
    elementwise_add(d_a, d_b, d_c)
cuda.Context.synchronize()

start = cuda.Event()
stop = cuda.Event()
start.record()
for _ in range(REPEATS):
    elementwise_add(d_a, d_b, d_c)
stop.record()
stop.synchronize()

elapsed_ms = start.time_till(stop)
mean_ms = elapsed_ms / REPEATS

# Retrieve and validate outside the event interval.
d_c.get(ary=h_c)
if not np.all(np.isfinite(h_c)):
    raise AssertionError('The GPU result contains nonfinite values.')
np.testing.assert_allclose(h_c, h_reference, rtol=1e-6, atol=1e-6)
print(f'Total device interval for {REPEATS} launches: {elapsed_ms:.6f} ms')
print(f'Mean device interval per launch: {mean_ms:.6f} ms')
print('Allocation, compilation, transfers and validation are excluded.')
print('Final result validated.')


Total device interval for 100 launches: 0.980960 ms
Mean device interval per launch: 0.009810 ms
Allocation, compilation, transfers and validation are excluded.
Final result validated.


## 11. Optional extension: a linear combination

The uploaded original example computes $c_i=\alpha a_i+\beta b_i$ with $\alpha=2$ and $\beta=3$. Keep this as a separate extension: add two scalar arguments and change only the per-element expression. Pass the coefficients as `np.float32` to match their declared `float` type.

Both products and their sum are expressed in one generated kernel, without separate full-size arrays for the two products. This illustrates why defining a custom expression can be useful beyond a simple sum; it does not establish a performance advantage by itself.

This cell overwrites `d_c` with the linear combination, validates it against the corresponding NumPy expression and prints selected values. Floating-point tolerance remains appropriate because CUDA may fuse a multiplication and an addition. To return to the ordinary sum, rerun the addition and retrieval cells.


In [12]:
linear_combination = ElementwiseKernel(
    "const float *a, const float *b, float *c, float alpha, float beta",
    "c[i] = alpha * a[i] + beta * b[i]",
    "linear_combination_elementwise",
)
alpha = np.float32(2.0)
beta = np.float32(3.0)
linear_combination(d_a, d_b, d_c, alpha, beta)
h_linear = d_c.get()
h_linear_reference = alpha * h_a + beta * h_b

if not np.all(np.isfinite(h_linear)):
    raise AssertionError('The GPU result contains nonfinite values.')
np.testing.assert_allclose(h_linear, h_linear_reference, rtol=1e-6, atol=1e-6)
print(f'Linear combination validated for all {N} elements: {alpha:g}*a + {beta:g}*b.')
print(f'Maximum absolute error: {np.max(np.abs(h_linear - h_linear_reference)):.3e}')
print(f"{'i':>8} {'a[i]':>12} {'b[i]':>12} {'2a+3b (GPU)':>15} {'NumPy':>12}")
for i in shown_indices:
    print(f'{i:8d} {h_a[i]:12.6f} {h_b[i]:12.6f} '
          f'{h_linear[i]:15.6f} {h_linear_reference[i]:12.6f}')


Linear combination validated for all 100003 elements: 2*a + 3*b.
Maximum absolute error: 0.000e+00
       i         a[i]         b[i]     2a+3b (GPU)        NumPy
       0     0.000000     0.000000        0.000000     0.000000
       1     0.250000     0.500000        2.000000     2.000000
       2     0.500000     1.000000        4.000000     4.000000
       3     0.750000     1.500000        6.000000     6.000000
       4     1.000000     2.000000        8.000000     8.000000
       5     1.250000     2.500000       10.000000    10.000000
       6     1.500000     3.000000       12.000000    12.000000
       7     1.750000     3.500000       14.000000    14.000000
       8     2.000000     4.000000       16.000000    16.000000
       9     2.250000     4.500000       18.000000    18.000000
      10     2.500000     5.000000       20.000000    20.000000
      11     2.750000     5.500000       22.000000    22.000000
  100000     2.500000    13.000000       44.000000    44.000000
  100

## 12. Release the GPU arrays

After completion, remove the references to the three device arrays. Their objects manage their allocations. To repeat the computation after this cell, start again from the transfer/allocation cell.


In [13]:
cuda.Context.synchronize()
del d_a, d_b, d_c
print('GPU array references released.')


GPU array references released.


## What changed?

The first two notebooks use a complete CUDA kernel, with explicit or automatic transfers. The third uses GPU array arithmetic. This fourth interface lets us customize the elementwise CUDA expression while PyCUDA supplies the surrounding kernel machinery. The same output buffer can be reused across calls.

**Execution status:** supplied with cleared outputs. Notebook structure, Python syntax and consistency of the host inputs have been checked. CUDA compilation and execution have not been performed in the preparation environment; run the notebook on your GPU to obtain results and timings.

Reference: [PyCUDA ElementwiseKernel documentation](https://documen.tician.de/pycuda/array.html#pycuda.elementwise.ElementwiseKernel).
